In [ ]:
%cd ../../

In [ ]:
import datetime

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl
import numpy as np
import holidays
from darts import TimeSeries
from pytorch_lightning.callbacks import TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
from darts import metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
from darts.models import CatBoostModel, TFTModel, TransformerModel, TSMixerModel, RNNModel, XGBModel, LightGBMModel, NBEATSModel
from darts.dataprocessing.transformers import Scaler, BoxCox, Diff
from sklearn.preprocessing import MinMaxScaler

In [ ]:
plt.style.use('seaborn-v0_8')
plt.rcParams.update({'font.size': 8})

# Load data and dims

## Load and process data

In [ ]:
EPS = 1e-6

In [ ]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

raw = []
for path in paths:
    df = pl.read_csv(path, separator=';', has_header=False, skip_rows=1)
    # df.columns = np.arange(df.shape[1], dtype=str)
    raw.append(df)


pos = pl.concat(raw)



# Rename columns
pos.columns = ['date', 'time', 'restaurant', 'meal_type', 'meal', 'pcs', 'co2']



# Convert pcs
pos = pos.filter((pl.col('pcs').is_not_null()) & (pl.col('pcs') >= 0))


# Map restaurant name
names_restaurant = {
    '600 Chemicum': 'che', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'exa', #'exactum'
    '570 Viikuna': 'vik',
}
pos = pos.with_columns(pl.col('restaurant').replace_strict(names_restaurant))


# Process date
pos = (
    pos
    .with_columns(
        (pl.col('date') + " " + pl.col('time')).str.to_datetime("%d.%m.%Y %H:%M").alias('datetime')
    )
    .drop('date', 'time')
)



# Accum by restaurant and date
pos = (
    pos
    .with_columns(
        pl.col('datetime').dt.date().alias('date')
    )

    .group_by('restaurant', 'date')
    .agg(pl.col('pcs').sum())
)


# Trim records out of the range
DATE_MIN = pl.lit('2023-01-09').str.to_date()
DATE_MAX = pl.lit('2025-03-31').str.to_date()
pos = (
    pos.filter(pl.col('date') >= DATE_MIN)

    # make it possible for MAPE and log
    .with_columns((pl.col('pcs') + EPS))
)

pos.head()

In [ ]:
pos_final = (
    pos
    .pivot(on='restaurant', index='date', values='pcs')
    .sort('date')
)

pos_final.head()

In [ ]:
restaurant = "che"

series = (
    TimeSeries
    .from_dataframe(
        pos_final.to_pandas(),
        time_col='date',
        value_cols=[
            restaurant
            # 'che',
            # 'exa',
            # 'phy',
            # 'vik'
        ],
        fillna_value=EPS,
        freq='B',
    )
)

series.plot()

In [ ]:
# static_covs_multi = pd.DataFrame(data={"restaurant": [0, 1, 2, 3]})
# series = series.with_static_covariates(static_covs_multi)
# series.static_covariates

## Load dims

### `dim_exam`

In [ ]:
dim_exam = (
    pl
    .from_records([
        {'date_begin': '2023-03-06', 'date_end': '2023-03-12'},  
        {'date_begin': '2023-05-01', 'date_end': '2023-05-07'},
        {'date_begin': '2023-10-23', 'date_end': '2023-10-29'},
        {'date_begin': '2023-12-18', 'date_end': '2023-12-24'},
        {'date_begin': '2024-03-04', 'date_end': '2024-03-10'},
        {'date_begin': '2024-05-06', 'date_end': '2024-05-12'},
        {'date_begin': '2024-10-21', 'date_end': '2024-10-27'},
        {'date_begin': '2025-03-03', 'date_end': '2025-03-09'},
    ])
    .with_columns(
        pl.col('date_begin').str.to_date(),
        pl.col('date_end').str.to_date(),
    )
)

dim_exam.head()

In [ ]:
df = (
    pl.DataFrame()

    # Create blank dataframe with date
    .with_columns(
        pl.date_range(DATE_MIN, DATE_MAX, '1d').alias('date')
    )
)

entries_exam = (
    df
    .join(dim_exam, how='cross')
    .filter(
        (pl.col('date') >= pl.col('date_begin'))
        & (pl.col('date') <= pl.col('date_end'))
    )
    .select(
        'date',
        pl.lit(1).alias('exam')
    )
)
cov_exam = (
    df
    .join(entries_exam, on='date', how='left')
    .with_columns(
        pl
        .col('exam')
        .fill_null(0)
        .cast(pl.String())
        .cast(pl.Categorical())
    )

    # Remove non-business days
    .filter(pl.col('date').dt.weekday() < 6)
)


cov_exam.head()

In [ ]:
series_exam = (
    TimeSeries
    .from_dataframe(
        df=cov_exam.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = False,
        value_cols='exam'
    )
    # .astype(np.float32)
)
series_exam.plot()

### `dim_holiday`

In [ ]:
fin_holidays = holidays.Finland(years=[2023, 2024, 2025])
dim_holiday = pl.from_records([
    {'date': d, 'name_holiday': n}
    for d, n in fin_holidays.items()
])

dim_holiday.head()

In [ ]:
cov_holiday = (
    pl.DataFrame()

    # Create blank dataframe with date
    .with_columns(
        pl.date_range(DATE_MIN, DATE_MAX, '1d').alias('date')
    )


    # Add dim holiday
    .join(dim_holiday, on='date', how='left')
    .with_columns(pl.col('name_holiday').is_not_null().cast(pl.Int32).alias('holiday'))
    .drop('name_holiday')


    # Cast to categorical type
    .with_columns(
        pl
        .col('holiday')
        .cast(pl.String())
        .cast(pl.Categorical())
    )

    # Remove non-business days
    .filter(pl.col('date').dt.weekday() < 6)
)

cov_holiday.head()


In [ ]:
series_holiday = (
    TimeSeries
    .from_dataframe(
        df=cov_holiday.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = False,
        value_cols='holiday'
    )
    # .astype(np.float32)
)
series_holiday.plot()

### `dim_bookings`

In [ ]:
dim_bookings = pl.read_parquet("data/inter/dim_booking_crafted.parquet")

cov_booking = (
    pl.DataFrame()

    # Create blank dataframe with date
    .with_columns(
        pl.date_range(DATE_MIN, DATE_MAX, '1d').alias('date')
    )


    # Add dim holiday
    .join(dim_bookings, on='date', how='left')
    .fill_null(0)

    # Remove non-business days
    .filter(pl.col('date').dt.weekday() < 6)

    # .drop
    .select('date', 'target_size', 'duration_coinciding', 'lecture_course', 'general_exam')
)
cov_booking.head()

In [ ]:
series_booking = (
    TimeSeries
    .from_dataframe(
        df=cov_booking.to_pandas(),
        time_col='date',
        freq='b',
        fill_missing_dates = False,
        value_cols=[
            'target_size',
            'duration_coinciding',
            # 'lecture_course',
            # 'general_exam'
        ]
    )
    # .astype(np.float32)
)



series_booking.plot()

In [ ]:
# transformer_cov_booking = Scaler(MinMaxScaler(feature_range=(-1, 1)))
# series_booking_transformed = transformer_cov_booking.fit_transform(series_booking)

# series_booking_transformed.plot()

# Train

In [ ]:
CUTOFF_DATE = pd.to_datetime("2024-10-01")
series_train, series_test = series.astype(np.float32).split_before(CUTOFF_DATE)

series_cov = (
    series_exam
    .concatenate(series_holiday, axis=1)
    # .concatenate(series_booking, axis=1)
    .astype(np.float32)
)

In [ ]:
# transformer = Diff(1, dropna=True)
# series_transformed = transformer.fit_transform(series)
# series_train_diff = series_transformed.drop_after(CUTOFF_DATE)


series_train_transformed = series_train
# series_train_transformed = series_train_diff

transformer_target = Scaler(MinMaxScaler(feature_range=(-1, 1)))
# transformer_target = BoxCox(lmbda=0.)
series_train_transformed = transformer_target.fit_transform(series_train_transformed)


series_train_transformed.plot()

## ML and statistical models

In [ ]:
version = datetime.datetime.now().strftime("%m-%d_%H-%M-%S")
model_name = "n-beats"

input_chunk_length = 5
output_chunk_length = len(series_test)
num_epochs = 500


# Define params
add_encoders = {
    "datetime_attribute": {
        "future": ["dayofweek", 'day', 'month'],
        "past": ["dayofweek", 'day', 'month']
    },
    'cyclic': {
        'past': ["dayofweek", 'day', 'month'],
        'future': ["dayofweek", 'day', 'month']
    },
}

params_ml = {
    "lags": input_chunk_length,
    "lags_future_covariates": [0],
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders}
}
params_dl = {
    "input_chunk_length": input_chunk_length,
    "output_chunk_length": output_chunk_length,
    "add_encoders": {**add_encoders},  
    "n_epochs": num_epochs,
    "pl_trainer_kwargs": {
        "callbacks": [TQDMProgressBar(refresh_rate=4)],
        "logger": [
            TensorBoardLogger("logs/tensorboard", name=f"{model_name}-{restaurant}", version=version, default_hp_metric=False)
        ],
        'precision': "32-true",
    },
    "optimizer_kwargs": {
        'lr': 5e-4
    }
}

# Define models
match model_name:
    # =================================================
    # ML models
    # =================================================
    case "catboost":
        model = CatBoostModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "xgboost":
        model = XGBModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "lightbgm":
        model = LightGBMModel(**params_ml)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )

    # =================================================
    # DL models
    # =================================================
    case 'rnn':
        params_dl['model'] = 'LSTM'

        model = RNNModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "tsmixer":
        model = TSMixerModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "transformer":
        model = TransformerModel(**params_dl)
        model.fit(
            series_train_transformed,
            past_covariates=series_cov.split_before(CUTOFF_DATE)[0],
        )
    case "tft":
        params_dl["categorical_embedding_sizes"] = {
            "holiday": (2, 2),
            "exam": (2, 2),
            "restaurant": (4, 4)
        }

        model = TFTModel(**params_dl)
        model.fit(
            series_train_transformed,
            future_covariates=series_cov,
        )
    case "n-beats":
        model = NBEATSModel(**params_dl)
        model.fit(
            series_train_transformed,
            past_covariates=series_cov.split_before(CUTOFF_DATE)[0],
        )
    
    case _:
        raise NotImplementedError()

### Test

In [ ]:
match model_name:
    # =================================================
    # ML models
    # =================================================
    case "catboost":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "xgboost":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "lightbgm":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )

    # =================================================
    # DL models
    # =================================================
    case "rnn":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "tsmixer":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "transformer":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov
        )
    case "transformer":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov
        )
    case "tft":
        series_pred = model.predict(
            n=len(series_test),
            future_covariates=series_cov
        )
    case "n-beats":
        series_pred = model.predict(
            n=len(series_test),
            past_covariates=series_cov
        )

    case _:
        raise NotImplementedError()
    

series_pred = transformer_target.inverse_transform(series_pred)
# series_pred = transformer.inverse_transform(series_train_diff.concatenate(series_pred)).drop_before(CUTOFF_DATE)

fig = plt.figure(figsize=(12, 12))

for idx, comp in enumerate(series_pred.components):
    ax = fig.add_subplot(2, 2, idx + 1)
    series_pred[comp].plot(ax=ax, label='pred')
    series_test[comp].plot(ax=ax, label='gt')
    # rmse_val = mape(series_test[comp], series_pred[comp])

    df = pd.concat(
        [
            series_test[comp].to_dataframe().rename(columns={comp: 'gt'}),
            series_pred[comp].to_dataframe().rename(columns={comp: 'pred'})
        ],
        axis=1
    )
    df = df[df['gt'] > EPS]
    mape_val = mean_absolute_percentage_error(df['gt'], df['pred'])

    ax.set_title(f"{comp} | mape = {mape_val:.4}")

# Train with entire data and forecast

In [ ]:
transformer_target = Scaler(MinMaxScaler(feature_range=(-1, 1)))
series_transformed = transformer_target.fit_transform(series)

model = CatBoostModel(**params)
model.fit(
    series_transformed,
    future_covariates=series_cov,
)

## Backtest

In [ ]:
result = model.historical_forecasts(series_transformed, start=.7, future_covariates=series_cov, forecast_horizon=1)
result = transformer_target.inverse_transform(result)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111)

series.plot(ax=ax, label='gt')
result.plot(ax=ax, label='pred')

## Forecast and save

In [ ]:
N = 107
series_pred = model.predict(n=N, future_covariates=series_cov)
series_pred = transformer_target.inverse_transform(series_pred)


series_pred.to_dataframe().to_excel("data/inter/evaluation/exactum_Apr1.xlsx")